# Klasifikasi Citra: Prediksi Sepatu Rapi atau Berantakan

**Notebook ini membangun sistem klasifikasi citra untuk menentukan apakah sebuah gambar sepatu termasuk kategori *Rapi* atau *Berantakan* menggunakan pendekatan Machine Learning berbasis fitur.**

---

## 1. Business Understanding

### Latar Belakang
Dalam kehidupan sehari-hari, kerapian penyimpanan sepatu mencerminkan kebiasaan dan kedisiplinan seseorang. Penilaian kerapian secara manual membutuhkan waktu dan bersifat subjektif. Oleh karena itu, dibutuhkan sebuah sistem otomatis berbasis **Computer Vision** yang mampu mengklasifikasikan kondisi sepatu secara objektif.

### Rumusan Masalah
- Bagaimana cara mengklasifikasikan gambar sepatu apakah termasuk **Rapi** atau **Berantakan** secara otomatis?
- Model Machine Learning mana yang memberikan **akurasi terbaik** tanpa mengalami *overfitting*?

### Tujuan
1. Membaca **semua format gambar** termasuk JPG, JPEG, PNG, BMP, TIFF, dan **HEIC**.
2. Mengekstrak fitur visual dari gambar sepatu (statistik warna RGB, Grayscale, dan tekstur GLCM).
3. Melatih dan membandingkan beberapa model klasifikasi (KNN, SVM, Random Forest, Gradient Boosting).
4. Memilih model terbaik berdasarkan akurasi test, train-test gap, dan metrik evaluasi lainnya.
5. Menyimpan model terbaik untuk kebutuhan deployment.

### Manfaat
- Dapat diintegrasikan pada sistem monitoring kerapian ruangan atau toko sepatu.
- Mendukung berbagai format gambar termasuk format kamera iPhone (HEIC).
- Memberikan umpan balik otomatis kepada pengguna terkait kondisi penyimpanan sepatu mereka.

## 2. Data Understanding

### Deskripsi Dataset
Dataset terdiri dari gambar sepatu yang terbagi menjadi dua kelas:
- **Rapi**: Gambar sepatu yang tersusun dengan teratur.
- **Berantakan**: Gambar sepatu yang tersusun tidak beraturan.

**Format yang didukung:** `.jpg`, `.jpeg`, `.png`, `.bmp`, `.tiff`, `.heic`, `.heif`

### Fitur yang Diekstrak

| No | Fitur | Deskripsi |
|---|---|---|
| 1-3 | Mean R, G, B | Rata-rata intensitas warna per channel |
| 4-6 | Std R, G, B | Standar deviasi intensitas warna per channel |
| 7-9 | Skewness R, G, B | Kemencengan distribusi intensitas per channel |
| 10 | Mean Gray | Rata-rata intensitas grayscale |
| 11 | Std Gray | Standar deviasi grayscale |
| 12 | Skewness Gray | Kemencengan grayscale |
| 13 | GLCM Energy | Energi tekstur (keseragaman) |
| 14 | GLCM Entropy | Entropi tekstur (kompleksitas) |
| 15 | GLCM Contrast | Kontras tekstur (variasi lokal) |
| 16 | GLCM Correlation | Korelasi tekstur |
| 17 | GLCM Homogeneity | Homogenitas tekstur |

## 3. Data Preparation

### 3.1 Instalasi & Import Library

In [ ]:
import subprocess, sys

packages = [
    'opencv-python',
    'scikit-image',
    'scikit-learn',
    'pandas',
    'numpy',
    'matplotlib',
    'seaborn',
    'joblib',
    'pillow',
    'pillow-heif'   # Support format HEIC/HEIF (kamera iPhone)
]
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('Semua library berhasil diinstal!')

In [ ]:
import cv2
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import skew
from skimage.feature import graycomatrix, graycoprops
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import joblib
from PIL import Image

# Aktifkan support HEIC/HEIF dari pillow-heif
try:
    from pillow_heif import register_heif_opener
    register_heif_opener()
    HEIC_SUPPORT = True
    print('✓ HEIC/HEIF support: AKTIF (pillow-heif terdeteksi)')
except ImportError:
    HEIC_SUPPORT = False
    print('⚠ HEIC/HEIF support: TIDAK AKTIF (pillow-heif tidak ditemukan, format lain tetap didukung)')

SEED = 42
np.random.seed(SEED)

print(f'\nOpenCV version : {cv2.__version__}')
print('Import selesai!')

### 3.2 Konfigurasi Path Dataset

In [ ]:
# Path dataset — sesuaikan jika perlu
BASE_DIR          = r'C:/Users/LENOVO/Documents/JP/psd/image_classification'
FOLDER_RAPI       = os.path.join(BASE_DIR, 'rapi')
FOLDER_BERANTAKAN = os.path.join(BASE_DIR, 'berantakan')
OUTPUT_CSV        = os.path.join(BASE_DIR, 'fitur_dataset_lengkap.csv')
MODEL_PATH        = os.path.join(BASE_DIR, 'model_terbaik_sepatu.pkl')
SCALER_PATH       = os.path.join(BASE_DIR, 'scaler_sepatu.pkl')

# Format gambar yang didukung (termasuk HEIC)
EXT_VALID = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif', '.heic', '.heif')

print('Konfigurasi Path:')
print(f'  Folder Rapi       : {FOLDER_RAPI}')
print(f'  Folder Berantakan : {FOLDER_BERANTAKAN}')
print(f'  Output CSV        : {OUTPUT_CSV}')
print(f'  Format didukung   : {EXT_VALID}')

### 3.3 Inventarisasi Dataset

In [ ]:
def inventarisasi_folder(folder_path, label):
    """Hitung dan tampilkan rincian file per ekstensi dalam satu folder."""
    if not os.path.exists(folder_path):
        print(f'[ERROR] Folder tidak ditemukan: {folder_path}')
        return []
    files = [f for f in os.listdir(folder_path) if f.lower().endswith(EXT_VALID)]
    ext_count = {}
    for f in files:
        ext = os.path.splitext(f)[1].lower()
        ext_count[ext] = ext_count.get(ext, 0) + 1
    return files, ext_count

files_rapi,       ext_rapi       = inventarisasi_folder(FOLDER_RAPI,       'rapi')
files_berantakan, ext_berantakan = inventarisasi_folder(FOLDER_BERANTAKAN, 'berantakan')

total_rapi       = len(files_rapi)
total_berantakan = len(files_berantakan)
total_dataset    = total_rapi + total_berantakan

print('=' * 55)
print('         INVENTARISASI DATASET SEPATU')
print('=' * 55)
print(f'  Kelas RAPI       : {total_rapi:>4} gambar')
for ext, cnt in ext_rapi.items():
    print(f'    {ext:>6} : {cnt} file')
print()
print(f'  Kelas BERANTAKAN : {total_berantakan:>4} gambar')
for ext, cnt in ext_berantakan.items():
    print(f'    {ext:>6} : {cnt} file')
print()
print(f'  TOTAL DATASET    : {total_dataset:>4} gambar')
print(f'  Balance ratio    : {total_rapi}/{total_berantakan} = {total_rapi/total_berantakan:.2f}')
print('=' * 55)

### 3.4 Fungsi Pembaca Gambar Universal (JPG / JPEG / PNG / HEIC)

> File HEIC (format kamera iPhone) dibaca menggunakan **pillow-heif**, lalu dikonversi ke array NumPy agar dapat diproses OpenCV.

In [ ]:
def read_image_universal(img_path):
    """
    Membaca gambar dari berbagai format: JPG, JPEG, PNG, BMP, TIFF, HEIC, HEIF.
    Mengembalikan array BGR (kompatibel OpenCV), atau None jika gagal.
    """
    ext = os.path.splitext(img_path)[1].lower()

    if ext in ('.heic', '.heif'):
        # Gunakan Pillow + pillow-heif untuk format HEIC
        try:
            pil_img = Image.open(img_path).convert('RGB')
            img_np  = np.array(pil_img)           # RGB array
            img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
            return img_bgr
        except Exception as e:
            print(f'  [HEIC ERROR] {os.path.basename(img_path)}: {e}')
            return None
    else:
        # Format standar → OpenCV
        img = cv2.imread(img_path)
        if img is None:
            # Fallback ke Pillow jika OpenCV gagal
            try:
                pil_img = Image.open(img_path).convert('RGB')
                img_np  = np.array(pil_img)
                return cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
            except Exception:
                return None
        return img


def extract_glcm_features(gray_img):
    """Ekstrak fitur tekstur GLCM dari citra grayscale."""
    gray_scaled = (gray_img / 4).astype(np.uint8)
    glcm = graycomatrix(
        gray_scaled,
        distances=[1],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=64,
        symmetric=True,
        normed=True
    )
    energy      = graycoprops(glcm, 'energy').mean()
    contrast    = graycoprops(glcm, 'contrast').mean()
    homogeneity = graycoprops(glcm, 'homogeneity').mean()
    correlation = graycoprops(glcm, 'correlation').mean()
    glcm_norm   = glcm[:, :, 0, 0] + 1e-10
    entropy     = -np.sum(glcm_norm * np.log2(glcm_norm))
    return energy, entropy, contrast, correlation, homogeneity


print('Fungsi pembaca gambar universal dan GLCM siap digunakan.')

### 3.5 Ekstraksi Fitur dari Semua Gambar

In [ ]:
def extract_all_features(folder_rapi, folder_berantakan):
    """Ekstrak semua fitur dari kedua folder dataset."""
    features = []
    skipped  = []
    folders  = {'rapi': folder_rapi, 'berantakan': folder_berantakan}

    for label, folder_path in folders.items():
        if not os.path.exists(folder_path):
            print(f'[PERINGATAN] Folder tidak ditemukan: {folder_path}')
            continue

        files = [f for f in os.listdir(folder_path) if f.lower().endswith(EXT_VALID)]
        print(f"Memproses '{label}': {len(files)} gambar...")
        berhasil = 0

        for filename in files:
            img_path = os.path.join(folder_path, filename)
            img = read_image_universal(img_path)

            if img is None:
                print(f'  [SKIP] Gagal membaca: {filename}')
                skipped.append(filename)
                continue

            # --- Fitur RGB ---
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ch_r, ch_g, ch_b = img_rgb[:,:,0], img_rgb[:,:,1], img_rgb[:,:,2]

            # --- Fitur Grayscale ---
            img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            # --- Fitur GLCM ---
            energy, entropy, contrast, correlation, homogeneity = extract_glcm_features(img_gray)

            features.append({
                'File_Name'          : filename,
                'Format'             : os.path.splitext(filename)[1].lower(),
                'Mean_R'             : np.mean(ch_r),
                'Mean_G'             : np.mean(ch_g),
                'Mean_B'             : np.mean(ch_b),
                'Std_R'              : np.std(ch_r),
                'Std_G'              : np.std(ch_g),
                'Std_B'              : np.std(ch_b),
                'Skew_R'             : skew(ch_r.flatten().astype(float)),
                'Skew_G'             : skew(ch_g.flatten().astype(float)),
                'Skew_B'             : skew(ch_b.flatten().astype(float)),
                'Mean_Gray'          : np.mean(img_gray),
                'Std_Gray'           : np.std(img_gray),
                'Skew_Gray'          : skew(img_gray.flatten().astype(float)),
                'GLCM_Energy'        : energy,
                'GLCM_Entropy'       : entropy,
                'GLCM_Contrast'      : contrast,
                'GLCM_Correlation'   : correlation,
                'GLCM_Homogeneity'   : homogeneity,
                'Label'              : label
            })
            berhasil += 1

        print(f"  → Berhasil: {berhasil}, Gagal: {len(files)-berhasil}")

    return pd.DataFrame(features), skipped


print('=' * 55)
print('  MEMULAI EKSTRAKSI FITUR DATASET SEPATU')
print('=' * 55)
df, file_gagal = extract_all_features(FOLDER_RAPI, FOLDER_BERANTAKAN)
df.to_csv(OUTPUT_CSV, index=False)

print(f'\nDataset tersimpan ke : {OUTPUT_CSV}')
if file_gagal:
    print(f'File gagal dibaca    : {file_gagal}')

### 3.6 Informasi Dataset Lengkap

In [ ]:
# Hitung split sebelum dilakukan (80% train, 20% test)
total     = len(df)
n_rapi    = (df['Label'] == 'rapi').sum()
n_beran   = (df['Label'] == 'berantakan').sum()
n_train   = int(total * 0.8)
n_test    = total - n_train

# Rincian per format
fmt_counts = df['Format'].value_counts()

print('╔══════════════════════════════════════════════╗')
print('║         INFORMASI DATASET SEPATU             ║')
print('╠══════════════════════════════════════════════╣')
print(f'║  Total Gambar        : {total:>5} gambar           ║')
print(f'║    - Kelas Rapi      : {n_rapi:>5} gambar           ║')
print(f'║    - Kelas Berantakan: {n_beran:>5} gambar           ║')
print('╠══════════════════════════════════════════════╣')
print('║  Format Gambar:                              ║')
for fmt, cnt in fmt_counts.items():
    heic_flag = ' (HEIC ✓)' if fmt in ('.heic','.heif') else ''
    print(f'║    {fmt:>6}: {cnt:>4} file{heic_flag:<17}║')
print('╠══════════════════════════════════════════════╣')
print(f'║  Split Data (80% / 20%)                      ║')
print(f'║    Data Latih (Train): {n_train:>5} gambar           ║')
print(f'║    Data Uji   (Test) : {n_test:>5}  gambar           ║')
print('╠══════════════════════════════════════════════╣')
print(f'║  Jumlah Fitur        : {df.shape[1]-3:>5} fitur             ║')
print(f'║  Missing Values      : {df.isnull().sum().sum():>5}                 ║')
print('╚══════════════════════════════════════════════╝')

### 3.7 Eksplorasi Data (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Distribusi kelas
kelas_count = df['Label'].value_counts()
colors_pie  = ['#42A5F5', '#EF5350']
wedges, texts, autotexts = axes[0].pie(
    kelas_count.values,
    labels=[f'{l.capitalize()}\n({v})' for l, v in zip(kelas_count.index, kelas_count.values)],
    autopct='%1.1f%%',
    colors=colors_pie,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title('Distribusi Kelas Dataset', fontweight='bold', fontsize=12)

# Plot 2: Jumlah gambar per format
fmt_df = df['Format'].value_counts().reset_index()
fmt_df.columns = ['Format', 'Jumlah']
bar_colors = ['#66BB6A' if f not in ('.heic','.heif') else '#FFA726' for f in fmt_df['Format']]
axes[1].bar(fmt_df['Format'], fmt_df['Jumlah'], color=bar_colors, edgecolor='white', alpha=0.9)
for i, (fmt, cnt) in fmt_df.iterrows():
    axes[1].text(i, cnt + 0.5, str(cnt), ha='center', fontweight='bold')
axes[1].set_title('Jumlah Gambar per Format', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Format')
axes[1].set_ylabel('Jumlah')
axes[1].grid(axis='y', alpha=0.3)

# Plot 3: Split Train/Test
split_labels = ['Train (80%)', 'Test (20%)']
split_values = [n_train, n_test]
split_colors = ['#7E57C2', '#26A69A']
bars = axes[2].bar(split_labels, split_values, color=split_colors, edgecolor='white', alpha=0.9, width=0.4)
for bar, val in zip(bars, split_values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 str(val), ha='center', fontsize=13, fontweight='bold')
axes[2].set_title('Pembagian Data Train / Test', fontweight='bold', fontsize=12)
axes[2].set_ylabel('Jumlah Gambar')
axes[2].set_ylim(0, max(split_values) * 1.2)
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('Ringkasan Dataset Sepatu', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'info_dataset.png'), dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Distribusi fitur utama per kelas
fitur_plot = ['Mean_R', 'Mean_G', 'Mean_B', 'Mean_Gray', 'GLCM_Energy', 'GLCM_Contrast']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, fitur in enumerate(fitur_plot):
    for label, color in [('rapi', '#42A5F5'), ('berantakan', '#EF5350')]:
        subset = df[df['Label'] == label][fitur]
        axes[i].hist(subset, bins=20, alpha=0.6, label=label.capitalize(), color=color, edgecolor='white')
    axes[i].set_title(f'Distribusi {fitur}', fontweight='bold')
    axes[i].set_xlabel(fitur)
    axes[i].set_ylabel('Frekuensi')
    axes[i].legend()
    axes[i].grid(alpha=0.3)

plt.suptitle('Distribusi Fitur: Sepatu Rapi vs Berantakan', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'distribusi_fitur.png'), dpi=100, bbox_inches='tight')
plt.show()

### 3.8 Pra-pemrosesan & Split Data

In [ ]:
fitur_cols = [c for c in df.columns if c not in ['File_Name', 'Format', 'Label']]
X = df[fitur_cols].values

le = LabelEncoder()  # berantakan=0, rapi=1
y  = le.fit_transform(df['Label'].values)
print(f'Pemetaan label: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Split 80% train / 20% test (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Normalisasi
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print()
print('╔══════════════════════════════════════════╗')
print('║      PEMBAGIAN DATA TRAIN / TEST         ║')
print('╠══════════════════════════════════════════╣')
print(f'║  Total Dataset : {len(X_train)+len(X_test):>4} gambar              ║')
print(f'║  Data Latih    : {len(X_train):>4} gambar (80%)         ║')
print(f'║    - Rapi      : {np.sum(y_train==1):>4} gambar              ║')
print(f'║    - Berantakan: {np.sum(y_train==0):>4} gambar              ║')
print(f'║  Data Uji      : {len(X_test):>4} gambar (20%)          ║')
print(f'║    - Rapi      : {np.sum(y_test==1):>4} gambar              ║')
print(f'║    - Berantakan: {np.sum(y_test==0):>4} gambar              ║')
print(f'║  Jumlah Fitur  : {X_train.shape[1]:>4} fitur               ║')
print('╚══════════════════════════════════════════╝')

## 4. Modeling

Empat model Machine Learning dilatih dan dibandingkan:
1. **K-Nearest Neighbors (KNN)** — Berbasis kedekatan jarak
2. **Support Vector Machine (SVM)** — Berbasis hyperplane margin maksimum
3. **Random Forest** — Ensemble berbasis bagging pohon keputusan
4. **Gradient Boosting** — Ensemble berbasis boosting bertahap

> **Kriteria model terbaik:** Akurasi test tertinggi + gap train-test ≤ 10% (tidak overfitting)

In [ ]:
models = {
    'KNN (k=5)'         : KNeighborsClassifier(n_neighbors=5, weights='distance', metric='euclidean'),
    'SVM (RBF)'         : SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=SEED),
    'Random Forest'     : RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_split=4,
                                                  random_state=SEED, class_weight='balanced'),
    'Gradient Boosting' : GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                                      max_depth=4, random_state=SEED)
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
hasil = {}

print('=' * 68)
print(f'{"Model":<20} {"Train":>8} {"CV (5-fold)":>12} {"Test":>8} {"Gap":>7} {"Status":>10}')
print('-' * 68)

for nama, model in models.items():
    model.fit(X_train, y_train)
    acc_train = accuracy_score(y_train, model.predict(X_train))
    acc_test  = accuracy_score(y_test,  model.predict(X_test))
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    acc_cv    = cv_scores.mean()
    gap       = acc_train - acc_test
    status    = '⚠ Overfit' if gap > 0.10 else '✓ OK'

    hasil[nama] = {
        'model'    : model,
        'acc_train': acc_train,
        'acc_cv'   : acc_cv,
        'acc_test' : acc_test,
        'gap'      : gap,
        'cv_scores': cv_scores,
        'y_pred'   : model.predict(X_test),
        'status'   : status
    }
    print(f'{nama:<20} {acc_train:>7.2%} {acc_cv:>11.2%} {acc_test:>7.2%} {gap:>+6.2%}  {status}')

print('=' * 68)

## 5. Evaluation

### 5.1 Rekapitulasi Akurasi & Pemilihan Model Terbaik

In [ ]:
# Pilih model terbaik: akurasi test tertinggi + tidak overfitting
kandidat = {n: v for n, v in hasil.items() if v['gap'] <= 0.10}
if not kandidat:
    kandidat = hasil  # fallback jika semua overfit

nama_terbaik  = max(kandidat, key=lambda k: kandidat[k]['acc_test'])
model_terbaik = kandidat[nama_terbaik]

# DataFrame perbandingan
df_hasil = pd.DataFrame([
    {
        'Model'              : n,
        'Train Acc'          : f"{v['acc_train']:.2%}",
        'CV Acc (5-fold)'    : f"{v['acc_cv']:.2%} (±{v['cv_scores'].std():.2%})",
        'Test Acc'           : f"{v['acc_test']:.2%}",
        'Gap Train-Test'     : f"{v['gap']:+.2%}",
        'Status'             : v['status']
    }
    for n, v in hasil.items()
]).sort_values('Test Acc', ascending=False).reset_index(drop=True)

print('Tabel Perbandingan Model:')
print(df_hasil.to_string(index=False))

print()
print('╔══════════════════════════════════════════════╗')
print(f'║  MODEL TERBAIK : {nama_terbaik:<29}║')
print('╠══════════════════════════════════════════════╣')
print(f'║  Akurasi Train : {model_terbaik["acc_train"]:.2%}                       ║')
print(f'║  Akurasi CV    : {model_terbaik["acc_cv"]:.2%} (±{model_terbaik["cv_scores"].std():.2%})           ║')
print(f'║  Akurasi Test  : {model_terbaik["acc_test"]:.2%}                       ║')
print(f'║  Gap Train-Test: {model_terbaik["gap"]:+.2%}                       ║')
print(f'║  Status        : {model_terbaik["status"]:<29}║')
print('╚══════════════════════════════════════════════╝')

In [ ]:
# Visualisasi perbandingan akurasi semua model
model_names    = list(hasil.keys())
acc_train_vals = [v['acc_train'] for v in hasil.values()]
acc_cv_vals    = [v['acc_cv']    for v in hasil.values()]
acc_test_vals  = [v['acc_test']  for v in hasil.values()]

x = np.arange(len(model_names))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width, acc_train_vals, width, label='Train',           color='#42A5F5', alpha=0.85)
bars2 = ax.bar(x,          acc_cv_vals,   width, label='Cross-Val (5-Fold)', color='#FFA726', alpha=0.85)
bars3 = ax.bar(x + width,  acc_test_vals, width, label='Test',            color='#66BB6A', alpha=0.85)

# Tandai model terbaik
idx_best = model_names.index(nama_terbaik)
ax.axvspan(idx_best - 0.4, idx_best + 0.4, alpha=0.08, color='gold', label=f'Model Terbaik: {nama_terbaik}')

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Akurasi', fontsize=12)
ax.set_title('Perbandingan Akurasi Model Klasifikasi\nSepatu Rapi vs Berantakan', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylim(0.5, 1.08)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.4)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.1%}', xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'perbandingan_model.png'), dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrix + CV Scores model terbaik
y_pred = model_terbaik['y_pred']
cm     = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Confusion Matrix — {nama_terbaik}', fontweight='bold')

axes[1].bar(range(1, 6), model_terbaik['cv_scores'], color='#42A5F5', alpha=0.8, edgecolor='white')
axes[1].axhline(y=model_terbaik['acc_cv'], color='red', linestyle='--',
                label=f"Rata-rata CV: {model_terbaik['acc_cv']:.2%}")
axes[1].set_xlabel('Fold ke-')
axes[1].set_ylabel('Akurasi')
axes[1].set_title(f'Cross-Validation (5-Fold) — {nama_terbaik}', fontweight='bold')
axes[1].set_ylim(0.6, 1.05)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
axes[1].legend()
axes[1].grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'evaluasi_model_terbaik.png'), dpi=100, bbox_inches='tight')
plt.show()

print(f'Classification Report — {nama_terbaik}')
print('=' * 50)
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Feature Importance (Random Forest / Gradient Boosting)
mdl_obj = model_terbaik['model']
if hasattr(mdl_obj, 'feature_importances_'):
    importances = mdl_obj.feature_importances_
    idx_sorted  = np.argsort(importances)[::-1]
    top_n = min(10, len(fitur_cols))

    plt.figure(figsize=(10, 5))
    plt.bar(range(top_n), importances[idx_sorted[:top_n]], color='#AB47BC', alpha=0.85)
    plt.xticks(range(top_n), [fitur_cols[i] for i in idx_sorted[:top_n]], rotation=30, ha='right')
    plt.xlabel('Nama Fitur')
    plt.ylabel('Tingkat Kepentingan')
    plt.title(f'Top-{top_n} Fitur Terpenting — {nama_terbaik}', fontweight='bold')
    plt.grid(axis='y', alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, 'fitur_terpenting.png'), dpi=100, bbox_inches='tight')
    plt.show()
else:
    print(f'[INFO] {nama_terbaik} tidak memiliki feature_importances_. Lewati visualisasi ini.')

## 6. Deployment

### 6.1 Simpan Model Terbaik

In [ ]:
joblib.dump(model_terbaik['model'], MODEL_PATH)
joblib.dump(scaler, SCALER_PATH)

print('✓ Model berhasil disimpan!')
print(f'  Model  : {MODEL_PATH}')
print(f'  Scaler : {SCALER_PATH}')

### 6.2 Fungsi Prediksi (Mendukung JPG / JPEG / PNG / HEIC)

In [ ]:
def predict_sepatu(image_path, model_path=MODEL_PATH, scaler_path=SCALER_PATH):
    """
    Prediksi apakah gambar sepatu RAPI atau BERANTAKAN.
    Mendukung: JPG, JPEG, PNG, BMP, TIFF, HEIC, HEIF.

    Returns: dict dengan 'prediksi' dan 'probabilitas'
    """
    model  = joblib.load(model_path)
    scaler = joblib.load(scaler_path)

    img = read_image_universal(image_path)
    if img is None:
        raise FileNotFoundError(f'Gambar tidak dapat dibaca: {image_path}')

    img_rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ch_r, ch_g, ch_b = img_rgb[:,:,0], img_rgb[:,:,1], img_rgb[:,:,2]

    energy, entropy, contrast, correlation, homogeneity = extract_glcm_features(img_gray)

    fitur_vektor = [
        np.mean(ch_r), np.mean(ch_g), np.mean(ch_b),
        np.std(ch_r),  np.std(ch_g),  np.std(ch_b),
        skew(ch_r.flatten().astype(float)),
        skew(ch_g.flatten().astype(float)),
        skew(ch_b.flatten().astype(float)),
        np.mean(img_gray), np.std(img_gray),
        skew(img_gray.flatten().astype(float)),
        energy, entropy, contrast, correlation, homogeneity
    ]

    X_input     = scaler.transform([fitur_vektor])
    pred_label  = model.predict(X_input)[0]
    pred_proba  = model.predict_proba(X_input)[0] if hasattr(model, 'predict_proba') else None

    label_map    = {0: 'Berantakan', 1: 'Rapi'}
    prediksi_str = label_map[pred_label]

    result = {'prediksi': prediksi_str}
    if pred_proba is not None:
        result['probabilitas'] = {
            'Berantakan': f'{pred_proba[0]:.2%}',
            'Rapi'       : f'{pred_proba[1]:.2%}'
        }
    return result

print('Fungsi predict_sepatu() siap! Mendukung JPG, PNG, HEIC, dan format lainnya.')

### 6.3 Demo Prediksi pada Gambar Contoh

In [ ]:
contoh_gambar = [
    (os.path.join(FOLDER_RAPI,       os.listdir(FOLDER_RAPI)[0]),       'Rapi'),
    (os.path.join(FOLDER_RAPI,       os.listdir(FOLDER_RAPI)[1]),       'Rapi'),
    (os.path.join(FOLDER_BERANTAKAN, os.listdir(FOLDER_BERANTAKAN)[0]), 'Berantakan'),
    (os.path.join(FOLDER_BERANTAKAN, os.listdir(FOLDER_BERANTAKAN)[1]), 'Berantakan'),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for col, (img_path, label_asli) in enumerate(contoh_gambar):
    hasil_pred = predict_sepatu(img_path)
    prediksi   = hasil_pred['prediksi']
    proba      = hasil_pred.get('probabilitas', {})
    benar      = (prediksi.lower() == label_asli.lower())

    img_show = read_image_universal(img_path)
    img_show = cv2.cvtColor(img_show, cv2.COLOR_BGR2RGB)
    img_show = cv2.resize(img_show, (224, 224))

    axes[0, col].imshow(img_show)
    axes[0, col].set_title(f'Label Asli: {label_asli}\n({os.path.splitext(img_path)[1]})',
                            fontsize=10, fontweight='bold')
    axes[0, col].axis('off')

    axes[1, col].axis('off')
    warna = '#2E7D32' if benar else '#C62828'
    info  = f'Prediksi: {prediksi}\nStatus: {"✓ Benar" if benar else "✗ Salah"}'
    if proba:
        info += f"\nP(Rapi): {proba['Rapi']}\nP(Berantakan): {proba['Berantakan']}"
    axes[1, col].text(0.5, 0.5, info, ha='center', va='center', fontsize=10,
                      transform=axes[1, col].transAxes,
                      bbox=dict(boxstyle='round,pad=0.5', facecolor=warna, alpha=0.2, edgecolor=warna))

plt.suptitle(
    f'Demo Prediksi — {nama_terbaik}\n'
    f'Dataset: {total_dataset} gambar | Train: {n_train} | Test: {n_test} | Akurasi Test: {model_terbaik["acc_test"]:.2%}',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'demo_prediksi.png'), dpi=100, bbox_inches='tight')
plt.show()

### 6.4 Ringkasan Akhir Proyek

In [ ]:
print('╔══════════════════════════════════════════════════════╗')
print('║           RINGKASAN PROYEK KLASIFIKASI SEPATU        ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  DATASET                                             ║')
print(f'║    Total Gambar      : {total_dataset:>4} gambar                   ║')
print(f'║    - Rapi            : {n_rapi:>4} gambar                   ║')
print(f'║    - Berantakan      : {n_beran:>4} gambar                   ║')
print(f'║    Format didukung   : JPG, JPEG, PNG, HEIC, dll.   ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  SPLIT DATA                                          ║')
print(f'║    Data Latih (80%)  : {n_train:>4} gambar                   ║')
print(f'║    Data Uji   (20%)  : {n_test:>4} gambar                    ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  FITUR                                               ║')
print(f'║    Jumlah Fitur      : {len(fitur_cols):>4} fitur                   ║')
print(f'║    (RGB + Grayscale + GLCM Tekstur)                  ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  MODEL TERBAIK : {nama_terbaik:<37}║')
print(f'║    Akurasi Train     : {model_terbaik["acc_train"]:.2%}                      ║')
print(f'║    Akurasi CV (5-fold): {model_terbaik["acc_cv"]:.2%} (±{model_terbaik["cv_scores"].std():.2%})          ║')
print(f'║    Akurasi Test      : {model_terbaik["acc_test"]:.2%}                      ║')
print(f'║    Gap Train-Test    : {model_terbaik["gap"]:+.2%}  → {model_terbaik["status"]}           ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  OUTPUT                                              ║')
print(f'║    Model   : model_terbaik_sepatu.pkl                ║')
print(f'║    Scaler  : scaler_sepatu.pkl                       ║')
print(f'║    Dataset : fitur_dataset_lengkap.csv               ║')
print('╚══════════════════════════════════════════════════════╝')